In [ ]:

#  Download / Load CSV file

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re


# ==============================
# Paths and Columns
# ==============================
file_path = "/content/drive/MyDrive/Bootcamp_ML_Data_Science/Capstone_Project/model_with_out_source_data/data_extraction/merged_shuffled_20250822_185836.csv"
target_col = "Label"

# read a small sample to inspect columns
df_head = pd.read_csv(file_path, nrows=5000)
print("Sample shape:", df_head.shape)
df_head.head()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==============================
# Choose Features
# ==============================
cols = df_head.columns.tolist()
assert target_col in cols, "Label column not found!"

# allow only 802.11 / radiotap / frame fields
allow_prefixes = ("frame.", "radiotap.", "wlan.", "wlan_radio.")

# hard blacklist
block_exact = set([
    "frame.number","frame.time","frame.time_epoch",
    "radiotap.mactime","radiotap.present.tsft","radiotap.timestamp.ts",
    "wlan.bssid","wlan.da","wlan.ra","wlan.sa","wlan.ta","wlan.ssid","wlan.tag","wlan.tag.length",
    "wlan.analysis.kck","wlan.analysis.kek","wlan.rsn.ie.gtk.key","wlan.rsn.ie.igtk.key","wlan.rsn.ie.pmkid",
    "wlan.fixed.timestamp","wlan_rsna_eapol.keydes.msgnr","wlan_rsna_eapol.keydes.data",
    "wlan_rsna_eapol.keydes.data_len","wlan_rsna_eapol.keydes.key_info.key_mic","wlan_rsna_eapol.keydes.nonce",
])

# regex blacklist
block_patterns = [
    r"(?i)\b(bssid|ssid|mac|addr|oui|vendor|station|ra|ta|sa|da)\b",
    r"(?i)^(ip\.|ipv6\.|arp|tcp\.|udp\.|dns|http|json|ssh|tls|smb2?|nbns|nbss|ldap|dhcp|mdns|ssdp)\b",
    r"(?i)(payload|data\.data|llc|eapol|key|nonce|pmkid|gtk|igtk|kck|kek)",
    r"(?i)(pcap|source_file|capture|interface)",
    r"(?i)(start_tsf|end_tsf|timestamp)",
]

def allowed(col: str) -> bool:
    if col == target_col:
        return True
    low = col.lower()
    if not low.startswith(allow_prefixes):
        return False
    if col in block_exact:
        return False
    for pat in block_patterns:
        if re.search(pat, low):
            return False
    return True

keep_cols = [c for c in cols if allowed(c)]
if target_col not in keep_cols:
    keep_cols.append(target_col)

print("Columns before:", len(cols), "| after filtering:", len(keep_cols))
print("First 15 kept:", keep_cols[:15])

df = pd.read_csv(file_path, usecols=keep_cols)
df.replace("?", np.nan, inplace=True)

print("Filtered dataset shape:", df.shape)
df.head(3)


In [ ]:
# ==============================
# Attack Classes of Interest
# ==============================
REALTIME_ATTACKS = ["SSDP", "Evil_Twin", "Krack", "Deauth", "RogueAP", "(Re)Assoc"]
df_plot = df[df[target_col].isin(REALTIME_ATTACKS)].reset_index(drop=True)

print("Dataset shape after filtering attacks:", df_plot.shape)
print("Unique classes:", df_plot[target_col].unique())

# Bar plot of counts
attack_counts = df_plot[target_col].value_counts()
plt.figure(figsize=(9,5))
sns.barplot(x=attack_counts.index, y=attack_counts.values)
plt.title("Attack Distribution (raw)")
plt.xlabel("Attack")
plt.ylabel("Rows")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

print("Counts:\n", attack_counts)


In [ ]:
# ==============================
# Check Null (Missing) Values
# ==============================
null_cnt = df_head.isna().sum().sort_values(ascending=False)
print("Top missing values columns:")
print(null_cnt.head(20))

In [ ]:
# ==============================
# Check Duplicates
# ==============================
dup_count = df_head.duplicated().sum()
print(f"Duplicate rows: {dup_count}")

In [ ]:
# ==============================
# Check Outliers (IQR method)
# ==============================
num_cols = df_head.select_dtypes(include=[np.number]).columns

outlier_report = {}
for c in num_cols:
    q1, q3 = df_head[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        continue
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    out_rate = ((df_head[c] < lower) | (df_head[c] > upper)).mean() * 100
    outlier_report[c] = out_rate

outlier_df = pd.Series(outlier_report).sort_values(ascending=False)
print("Outlier percentage (top 15):")
display(outlier_df.head(15))

In [ ]:
# ==============================
# Correlation Heatmap
# ==============================
num_cols = df.select_dtypes(include=[np.number]).columns
print(num_cols)
topN = min(20, len(num_cols))  # show up to 20 numeric cols
corr = df[num_cols[:topN]].corr(method="spearman")

plt.figure(figsize=(12,8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Spearman)")
plt.show()

In [ ]:
# ==============================
# Profiling Report (ydata-profiling)
# ==============================
!pip install ydata-profiling

from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Wi-Fi Attacks Data Profiling", explorative=True)

# Show in notebook
profile.to_notebook_iframe()

# Or save to file
profile.to_file("profiling_report.html")

# The selected columns for the ML model are:
---
# 📊 Feature Categorization for Attack Detection

## ⏱️ Timing & Traffic Patterns
| Feature | Description | Why it Matters for Attack Detection |
|----------|-------------|-------------------------------------|
| frame.time_relative | Time since start of capture | Detects abnormal activity bursts (DoS, flooding). |
| frame.time_delta | Time gap between frames | Helps spot replay or deauth floods (very small gaps). |
| frame.time_delta_displayed | Displayed delta between frames | Similar to above, useful for fine-grained timing anomalies. |
| wlan.duration | Medium reservation duration | Abnormal values may indicate crafted frames in DoS attacks. |
| wlan_radio.duration | PHY frame duration | Helps detect oversized/malformed packets in attacks. |

---

## 📦 Frame Size & Structure
| Feature | Description | Why it Matters for Attack Detection |
|----------|-------------|-------------------------------------|
| frame.len | Frame length | Oversized/malformed packets may signal fuzzing or injection attacks. |
| frame.encap_type | Link-layer encapsulation | Non-standard encapsulation can indicate rogue tools. |

---

## 📡 Channel & Frequency
| Feature | Description | Why it Matters for Attack Detection |
|----------|-------------|-------------------------------------|
| radiotap.channel.freq | Channel frequency (MHz) | Rogue APs/Evil Twins often appear on nearby frequencies. |
| wlan_radio.channel | Reported channel | Cross-check with expected channel. |
| wlan_radio.frequency | Radio frequency (MHz) | Helps detect channel hopping attacks. |
| wlan.country_info.fnm | Country info (freq mgmt) | Attackers may spoof regulatory info. |
| wlan.country_info.code | Country code (e.g., US, EU) | Misleading codes may indicate rogue APs. |
| radiotap.channel.flags.ofdm | OFDM modulation flag | Detects PHY manipulation. |
| radiotap.channel.flags.cck | CCK modulation flag | Used to identify modulation anomalies. |

---

## 📶 Signal Strength & PHY Metrics
| Feature | Description | Why it Matters for Attack Detection |
|----------|-------------|-------------------------------------|
| radiotap.dbm_antsignal | Signal strength (dBm) | Sudden strong signals = nearby Evil Twin / rogue AP. |
| wlan_radio.signal_dbm | Reported radio signal strength | Cross-checks physical location anomalies. |
| radiotap.datarate | PHY data rate (Mbps) | Abnormally low/high rates can be suspicious. |
| wlan_radio.data_rate | Reported radio rate | Detects spoofed rates in jamming/Evil Twin. |
| radiotap.length | Radiotap header length | Out-of-spec headers may indicate crafted packets. |
| wlan_radio.phy | PHY type (802.11a/b/g/n/ac) | Unexpected PHY may indicate rogue injection. |

---

## 📑 Frame Control Flags
| Feature | Description | Why it Matters for Attack Detection |
|----------|-------------|-------------------------------------|
| wlan.fc.type | Frame type (Mgmt, Control, Data) | Attackers may craft fake management frames. |
| wlan.fc.subtype | Subtype (Beacon, Probe, Deauth, etc.) | Directly tied to attacks (e.g., Deauth subtype = 0x0c). |
| wlan.fc.ds | Distribution system flag | Reveals frame direction (to/from DS). |
| wlan.fc.protected | Encryption flag | Unprotected sensitive frames → security risk. |
| wlan.fc.pwrmgt | Power management flag | Misuse in DoS or spoofing. |
| wlan.fc.frag | Fragment flag | Used in fragmentation attacks. |
| wlan.fc.order | Strict ordering flag | Rarely used, anomalies suggest crafted packets. |
| wlan.fc.moredata | More data buffered flag | Abnormal set can indicate spoofing. |
| wlan.fc.retry | Retry bit | High retries = interference or forced retransmission attack. |

---

## 🔑 Identifiers
| Feature | Description | Why it Matters for Attack Detection |
|----------|-------------|-------------------------------------|
| wlan.seq | Sequence number | Replay attacks or injection produce anomalies. |


In [ ]:
# Cell 4 — STEP 3: Filter to Normal + selected attacks, map to binary (0/1)
selected_attacks = ["SSDP", "Evil_Twin", "Krack", "Deauth", "(Re)Assoc", "RogueAP"]
mask = (df[target_col] == "Normal") | (df[target_col].isin(REALTIME_ATTACKS))
df = df.loc[mask].copy()

# Binary target: 0 = Normal, 1 = Attack
df["Binary_Label"] = (df[target_col] != "Normal").astype(int)

# Separate features/target
feature_cols = [c for c in df.columns if c not in [target_col, "Binary_Label"]]
X = df[feature_cols].copy()
y = df["Binary_Label"].copy()

print("Class counts (overall):")
print(y.value_counts().rename({0: "Normal", 1: "Attack"}))

In [ ]:
# Cell ROS-1 — Install imbalanced-learn (run once if needed)
# !pip install imbalanced-learn
from imblearn.over_sampling import RandomOverSampler

# Target 85% Normal / 15% Attack:
desired_attack_prop = 0.15
ratio = desired_attack_prop / (1 - desired_attack_prop)  # ≈ 0.17647

ros = RandomOverSampler(sampling_strategy=ratio, random_state=42)
X_bal, y_bal = ros.fit_resample(X, y)

print("Overall counts AFTER ROS to ~85/15:")
print(y_bal.value_counts().rename({0: "Normal", 1: "Attack"}))


In [ ]:
# Cell SPLIT — Train/Test split AFTER ROS (keep stratification)
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

print("Train counts:")
print(y_tr.value_counts().rename({0: "Normal", 1: "Attack"}))
print("\nTest counts:")
print(y_te.value_counts().rename({0: "Normal", 1: "Attack"}))


In [ ]:
# Cell PRE-5 — Coerce mixed columns (numeric vs categorical) + drop almost-all-NaN
import pandas as pd
import numpy as np
import re

def coerce_mixed_columns(df_train, df_test, hex_ok=True, thresh=0.8):
    df_train = df_train.copy()
    df_test  = df_test.copy()
    num_cols, cat_cols = [], []
    for c in df_train.columns:
        s_tr, s_te = df_train[c], df_test[c]
        if s_tr.dtype == object or str(s_tr.dtype).startswith("string"):
            st_tr = s_tr.astype(str)
            if hex_ok:
                st_tr_num = pd.to_numeric(st_tr.str.replace(r"^\s*0x", "", regex=True), errors="coerce")
            else:
                st_tr_num = pd.to_numeric(st_tr, errors="coerce")
            if st_tr_num.notna().mean() >= thresh:
                st_te = s_te.astype(str)
                st_te_num = pd.to_numeric(st_te.str.replace(r"^\s*0x", "", regex=True), errors="coerce")
                df_train[c] = st_tr_num.astype("float32")
                df_test[c]  = st_te_num.astype("float32")
                num_cols.append(c)
            else:
                df_train[c] = st_tr.astype("string")
                df_test[c]  = s_te.astype("string")
                cat_cols.append(c)
        else:
            num_cols.append(c)
    return df_train, df_test, num_cols, cat_cols

X_tr, X_te, num_cols, cat_cols = coerce_mixed_columns(X_tr, X_te, hex_ok=True, thresh=0.8)

# Drop columns that are almost all NaN in TRAIN (mirror on TEST)
na_ratio = X_tr.isna().mean()
drop_almost = na_ratio[na_ratio > 0.98].index.tolist()
if drop_almost:
    X_tr.drop(columns=drop_almost, inplace=True, errors="ignore")
    X_te.drop(columns=[c for c in drop_almost if c in X_te.columns], inplace=True, errors="ignore")
    num_cols = [c for c in num_cols if c not in drop_almost]
    cat_cols = [c for c in cat_cols if c not in drop_almost]

print("Numerical Features:", len(num_cols), "Categorical Features", len(cat_cols))
